First we need to analyze what predictor variables are significant,(not analyzing name, id, and its already grouped by coverage). also not using risk_tier becuase thats applied after underwriting. Model trained only on data morked training(not holdout)

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

freq   = pd.read_csv(PROC / "frequency_data.csv")   # 40,000 student-coverage rows
claims = pd.read_csv(PROC / "claims.csv")           # 1,819 individual claims

# 'class' is a Python keyword, so patsy formulas cannot use it
freq   = freq.rename(columns={'class': 'class_year'})
claims = claims.rename(columns={'class': 'class_year'})

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

# training only; holdout is sealed until validation
fq_train = {c: freq[(freq.coverage == c) &
                    (freq.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

cl_train = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

RATING_VARS = ['class_year', 'study', 'gpa', 'greek', 'gender',
               'off_campus', 'distance_to_campus', 'sprinklered']

pd.DataFrame({
    'exposures': {c: len(fq_train[c]) for c in COVERAGES},
    'claims':    {c: len(cl_train[c]) for c in COVERAGES},
    'frequency': {c: fq_train[c].claim_count.mean().round(4) for c in COVERAGES},
})

,exposures,claims,frequency
Personal Property,7999,787,0.0984
Additional Living Expense,7999,427,0.0534
Guest Medical,7999,187,0.0234
Liability,7999,67,0.0084


seperated training and holdout. renamed class to classyear to avoid python conflicts with class

Personal property

In [16]:
pp = fq_train['Personal Property'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp[c] = pp[c].astype('category')

pp.head(10)

,class_year,study,gpa,greek,gender,off_campus,distance_to_campus,sprinklered,claim_count
0,Junior,Humanities,1.203913,Non-greek,Male,On campus,0.000000,True,0
1,Junior,Science,0.609616,Non-greek,Female,On campus,0.000000,True,0
2,Junior,Business,3.073385,Non-greek,Female,Off campus,0.223942,False,0
3,Senior,Business,3.224736,Non-greek,Male,On campus,0.000000,False,0
4,Sophomore,Business,2.687851,Non-greek,Female,On campus,0.000000,False,0
5,Junior,Business,0.688659,Non-greek,Other,On campus,0.000000,True,0
6,Junior,Science,3.243341,Non-greek,Female,On campus,0.000000,False,0
7,Junior,Humanities,3.317148,Non-greek,Female,Off campus,0.467159,True,0
8,Freshman,Business,1.309350,Greek,Other,On campus,0.000000,True,0
9,Sophomore,Other,0.981003,Non-greek,Female,Off campus,10.083255,True,1


got relevant data and added 

In [17]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# holdout frame for Personal Property, same treatment as the training frame
pp_hold = freq[(freq.coverage == 'Personal Property') &
               (freq.training_holdout == 'holdout')].copy()
for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp_hold[c] = pp_hold[c].astype('category')


def poisson_deviance(y, mu):
    """2 * sum[ y*log(y/mu) - (y - mu) ], with 0*log(0) = 0."""
    y   = np.asarray(y,  dtype=float)
    mu  = np.asarray(mu, dtype=float)
    pos = y > 0
    term = np.zeros_like(y)
    term[pos] = y[pos] * np.log(y[pos] / mu[pos])
    return 2.0 * np.sum(term - (y - mu))


rows = []

# baseline: intercept only
null = smf.glm('claim_count ~ 1', data=pp, family=sm.families.Poisson()).fit()
rows.append(('(null)', null.aic,
             poisson_deviance(pp_hold.claim_count, null.predict(pp_hold))))

# one model per candidate predictor
for v in RATING_VARS:
    m = smf.glm(f'claim_count ~ {v}', data=pp, family=sm.families.Poisson()).fit()
    rows.append((v, m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step1 = (pd.DataFrame(rows, columns=['predictor', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step1.round(3)

,predictor,train_AIC,holdout_deviance
0,class_year,5282.815,912.590
1,off_campus,5279.606,905.648
2,distance_to_campus,5285.333,904.860
3,study,5292.084,903.980
4,gender,5293.265,903.975
5,(null),5289.290,903.764
6,sprinklered,5291.084,903.739
7,gpa,5289.773,903.381
8,greek,5188.615,895.082


Greek is going to be the first predictor

Now to check all 2 parameter models

In [18]:
rows = []

base = smf.glm('claim_count ~ greek', data=pp, family=sm.families.Poisson()).fit()
rows.append(('greek', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x != 'greek']:
    f = f'claim_count ~ greek + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step2 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step2.round(3)

,model,train_AIC,holdout_deviance
0,greek + class_year,5183.433,903.297
1,greek + off_campus,5177.649,897.045
2,greek + distance_to_campus,5183.982,896.588
3,greek + study,5190.051,895.812
4,greek + gender,5192.647,895.303
5,greek,5188.615,895.082
6,greek + gpa,5189.088,895.047
7,greek + sprinklered,5190.363,894.928


moving foward with greek and sprinklered

In [19]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered')]:
    f = f'claim_count ~ greek + sprinklered + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step3 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step3.round(3)

,model,train_AIC,holdout_deviance
0,greek + sprinklered + class_year,5185.212,903.141
1,greek + sprinklered + off_campus,5179.408,896.845
2,greek + sprinklered + distance_to_campus,5185.772,896.408
3,greek + sprinklered + study,5191.800,895.664
4,greek + sprinklered + gender,5194.384,895.134
5,greek + sprinklered,5190.363,894.928
6,greek + sprinklered + gpa,5190.861,894.885


Analyze all 4 predictor models

In [ ]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered + gpa', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered + gpa', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered', 'gpa')]:
    f = f'claim_count ~ greek + sprinklered + gpa + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + gpa + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step4 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step4.round(3)

,model,train_AIC,holdout_deviance
0,greek + sprinklered + gpa + class_year,5185.639,903.190
1,greek + sprinklered + gpa + off_campus,5179.912,896.806
2,greek + sprinklered + gpa + distance_to_campus,5186.298,896.332
3,greek + sprinklered + gpa + study,5192.313,895.678
4,greek + sprinklered + gpa + gender,5194.786,895.114
5,greek + sprinklered + gpa,5190.861,894.885


**Since the 3 predictor model has lower test deviance, we will use these 3 predictors to analyze personal property

Now for Additional Living Expenses

In [ ]:
ale = fq_train['Additional Living Expense'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale[c] = ale[c].astype('category')

ale_hold = freq[(freq.coverage == 'Additional Living Expense') &
                (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale_hold[c] = ale_hold[c].astype('category')

,class_year,study,gpa,greek,off_campus,distance_to_campus,sprinklered,claim_count
0,Junior,Humanities,1.203913,Non-greek,On campus,0.000000,True,0
1,Junior,Science,0.609616,Non-greek,On campus,0.000000,True,0
2,Junior,Business,3.073385,Non-greek,Off campus,0.223942,False,0
3,Senior,Business,3.224736,Non-greek,On campus,0.000000,False,0
4,Sophomore,Business,2.687851,Non-greek,On campus,0.000000,False,0
5,Junior,Business,0.688659,Non-greek,On campus,0.000000,True,0
6,Junior,Science,3.243341,Non-greek,On campus,0.000000,False,0
7,Junior,Humanities,3.317148,Non-greek,Off campus,0.467159,True,0
8,Freshman,Business,1.309350,Greek,On campus,0.000000,True,0
9,Sophomore,Other,0.981003,Non-greek,Off campus,10.083255,True,0


First analyzing all 1 predictor models

In [24]:
def forward_stepwise(train, hold, candidates, min_gain=2.0):
    """Forward selection on out-of-sample Poisson deviance.
    Returns (selected variables, one combined table of every step)."""
    keep     = []
    m_cur    = smf.glm('claim_count ~ 1', data=train,
                       family=sm.families.Poisson()).fit()
    d_cur    = poisson_deviance(hold.claim_count, m_cur.predict(hold))
    all_rows = []
    step     = 0

    while True:
        remaining = [v for v in candidates if v not in keep]
        if not remaining:
            break
        step += 1

        rows = [{'step': step, 'added': '-',
                 'model': ' + '.join(keep) or '(null)',
                 'train_AIC': m_cur.aic, 'holdout_deviance': d_cur,
                 'note': 'current'}]

        results = {}
        for v in remaining:
            f = 'claim_count ~ ' + ' + '.join(keep + [v])
            m = smf.glm(f, data=train, family=sm.families.Poisson()).fit()
            d = poisson_deviance(hold.claim_count, m.predict(hold))
            results[v] = (m, d)
            rows.append({'step': step, 'added': v,
                         'model': ' + '.join(keep + [v]),
                         'train_AIC': m.aic, 'holdout_deviance': d, 'note': ''})

        best = min(results, key=lambda v: results[v][1])
        gain = d_cur - results[best][1]

        for r in rows:
            if r['added'] == best:
                r['note'] = ('SELECTED' if gain >= min_gain
                             else f'best, gain {gain:.2f} < {min_gain}')

        all_rows.extend(sorted(rows, key=lambda r: r['holdout_deviance']))

        if gain < min_gain:
            break

        keep.append(best)
        m_cur, d_cur = results[best]

    return keep, pd.DataFrame(all_rows)

Not showing table ^ need to fix

Now analyzing 2 predictor models with greek

In [13]:
rows = []

base = smf.glm('claim_count ~ greek', data=ale, family=sm.families.Poisson()).fit()
rows.append(('greek', base.aic,
             poisson_deviance(ale_hold.claim_count, base.predict(ale_hold))))

for v in [x for x in RATING_VARS if x != 'greek']:
    f = f'claim_count ~ greek + {v}'
    m = smf.glm(f, data=ale, family=sm.families.Poisson()).fit()
    rows.append((f'greek + {v}', m.aic,
                 poisson_deviance(ale_hold.claim_count, m.predict(ale_hold))))

ale_step2 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
               .sort_values('holdout_deviance', ascending=False)
               .reset_index(drop=True))

ale_step2.round(3)

C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * n

,model,train_AIC,holdout_deviance
0,greek + study,3329.659,599.012
1,greek + class_year,3331.100,598.702
2,greek + off_campus,3322.583,597.095
3,greek + distance_to_campus,3326.055,596.843
4,greek + gpa,3326.860,595.789
5,greek,3326.521,595.637
6,greek + sprinklered,3327.548,594.177


Analyzing all 3 predictor models with greek and sprinklered

In [14]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered', data=ale,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered', base.aic,
             poisson_deviance(ale_hold.claim_count, base.predict(ale_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered')]:
    f = f'claim_count ~ greek + sprinklered + {v}'
    m = smf.glm(f, data=ale, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + {v}', m.aic,
                 poisson_deviance(ale_hold.claim_count, m.predict(ale_hold))))

ale_step3 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
               .sort_values('holdout_deviance', ascending=False)
               .reset_index(drop=True))

ale_step3.round(3)

C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_24616\3079082868.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * n

,model,train_AIC,holdout_deviance
0,greek + sprinklered + study,3330.644,597.538
1,greek + sprinklered + class_year,3332.160,597.208
2,greek + sprinklered + off_campus,3323.595,595.689
3,greek + sprinklered + distance_to_campus,3327.026,595.403
4,greek + sprinklered + gpa,3327.832,594.327
5,greek + sprinklered,3327.548,594.177


**Based on above, we only use greek and sprinklered for out additional living expenses model

Now to analyze Guest Medical